# RLM2 в Colab — свип размера модели (ETH или золото)

Runtime → Change runtime type: CPU, High-RAM (8 ядер). Затем Run all. В ячейке настроек выберите MARKET. Для золота загрузите `xauusd_m1.parquet` в `/content/ct/data/ml/gold/` (ячейка скачает его с Google Drive по ссылке пользователя). В конце скачается zip с результатами.

In [ ]:
MARKET='ETHUSDT'   # или 'XAUUSD'
import getpass,subprocess
tok=getpass.getpass('GitHub token (Enter, если репозиторий публичный): ')
url=f'https://{tok}@github.com/Flyingzaptop/crypto-trend.git' if tok else 'https://github.com/Flyingzaptop/crypto-trend.git'
subprocess.run(['git','clone','-q','-b','claude/drawdowns-ml',url,'/content/ct'],check=True)
%cd /content/ct

In [ ]:
!pip -q install stable-baselines3 gymnasium lightgbm pyarrow

In [ ]:
if MARKET=='XAUUSD':
    !mkdir -p data/ml/gold && curl -sSL -o data/ml/gold/xauusd_m1.parquet 'https://drive.usercontent.google.com/download?id=1S3AjTWTWzq76bUyHxlgTqNhZzVsOtgwf&export=download&confirm=t'
else:
    !python data_m1.py ETHUSDT BTCUSDT

In [ ]:
import os
n=os.cpu_count()
!python -W ignore rlm2.py --market {MARKET} --jobs {n}

In [ ]:
!zip -qr rlm2_{MARKET}.zip docs/ml/results/rlm2_runs
from google.colab import files
files.download(f'rlm2_{MARKET}.zip')